# 💧 HydroForecast Seasonal Volume Calculator

Calculates cumulative forecasted volumes from Streamlit forecast CSVs and produces:

1. **Validation Plot** — 30-day monthly volumes in **cubic feet**, matching the Streamlit reference tool
2. **Cumulative Volume Plot** — Volumes summed over a user-defined look-ahead window in **acre-feet**

> Once Plot 1 validates against Streamlit, Plot 2 can be trusted for deeper analysis.

---
### How the calculation works

Each row in the CSV is the average flow rate (CFS) for one lead-time step.
The step size (e.g. 24 h or 240 h) is **auto-detected** from your CSV.

**Volume per step (cubic feet)** = `flow_CFS × 864,000`

- **Validation plot:** auto-generates for all months using `lead_time ≤ 720 h − step_size` (matches Streamlit)
- **Seasonal plot:** sums `FORECAST_HORIZON_MONTHS` months of lead-time per issue date, across `START` → `END` window
- Both divide by **43,560** to convert cubic feet → acre-feet (Plot 2 only)


## 1. Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "pandas", "openpyxl", "-q"])


In [ ]:
import pandas as pd
import calendar
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
import io


## 2. Upload Forecast CSV

Export from the Streamlit tool using **"Download Forecasts and Obs as CSV"** and upload here.

**Required columns:** `issue_time`, `lead_time`, `forecast`, `reference_forecast`, `obs`


In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

# ── Validate required columns ─────────────────────────────────────────────────
REQUIRED_COLS = ['issue_time', 'lead_time', 'forecast', 'reference_forecast', 'obs']
missing = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing:
    raise ValueError(f"❌ Missing required columns: {missing}\nFound: {df_raw.columns.tolist()}")

# ── Parse issue_time robustly (handles UTC offset present or absent) ──────────
try:
    df_raw['issue_time'] = pd.to_datetime(df_raw['issue_time'], utc=True)
except Exception:
    df_raw['issue_time'] = pd.to_datetime(df_raw['issue_time']).dt.tz_localize('UTC')

df_raw['Day']   = df_raw['issue_time'].dt.day
df_raw['Month'] = df_raw['issue_time'].dt.month
df_raw['Year']  = df_raw['issue_time'].dt.year

# ── Handle multiple drainage areas ───────────────────────────────────────────
if 'drainage_area_feature_id' in df_raw.columns:
    site_ids = df_raw['drainage_area_feature_id'].unique()
    if len(site_ids) > 1:
        print(f"⚠️  Multiple sites found: {site_ids}")
        print(f"   Using first site: {site_ids[0]}")
        print(f"   To use a different site, set SITE_ID below and re-run this cell.")
    SITE_ID = site_ids[0]
    df = df_raw[df_raw['drainage_area_feature_id'] == SITE_ID].copy()
else:
    df = df_raw.copy()
    SITE_ID = None

# ── Auto-detect lead-time step size ──────────────────────────────────────────
lead_times_sorted = sorted(df['lead_time'].unique())
if len(lead_times_sorted) < 2:
    raise ValueError("❌ Need at least 2 distinct lead_time values to detect step size.")
STEP_SIZE_HOURS = int(min(
    lead_times_sorted[i+1] - lead_times_sorted[i]
    for i in range(len(lead_times_sorted) - 1)
))
VALIDATION_LEAD_MAX = 720 - STEP_SIZE_HOURS  # 30-day window matching Streamlit

print(f"✅ Loaded {len(df):,} rows from '{filename}'")
print(f"   Date range     : {df['issue_time'].min().date()} → {df['issue_time'].max().date()}")
print(f"   Site ID        : {SITE_ID}")
print(f"   Lead-time step : {STEP_SIZE_HOURS}h (auto-detected)")
print(f"   Plot 1 lead max: {VALIDATION_LEAD_MAX}h  (= 720 − {STEP_SIZE_HOURS})")
print(f"   Years available: {sorted(df['Year'].unique())}")


## 3. User Inputs ✏️

Edit only the block below — everything else updates automatically.

| Parameter | Description |
|-----------|-------------|
| `START_YEAR` / `START_MONTH` | Start of the analysis window |
| `END_YEAR` / `END_MONTH` | End of the analysis window |
| `FORECAST_DAY` | Day of month the forecast was issued (typically `1`) |
| `FORECAST_HORIZON_MONTHS` | Number of months of lead time summed per bar in Plot 2 |

**Example:** `START_YEAR=2025`, `START_MONTH=8`, `END_YEAR=2026`, `END_MONTH=4` → Aug 2025 – Apr 2026

> **Plot 1** always auto-generates for the full date window — `FORECAST_HORIZON_MONTHS` does not affect it.


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║             USER INPUTS — EDIT HERE              ║
# ╚══════════════════════════════════════════════════╝

START_YEAR   = 2025   # Start year of analysis window
START_MONTH  = 8      # Start month of analysis window (1 = January)
END_YEAR     = 2026   # End year of analysis window
END_MONTH    = 4      # End month of analysis window

FORECAST_DAY            = 1   # Day of month forecast was issued (typically 1)
FORECAST_HORIZON_MONTHS = 3   # Months of lead time summed per bar in Plot 2

# ── Fixed conversions — do not edit ──────────────────────────────────────────
CFS_TO_CF_PER_STEP = 864_000   # 1 CFS × 10 days × 86,400 s/day = cubic feet per step
CF_TO_ACRE_FEET    = 43_560    # cubic feet per acre-foot


## 4. Input Validation

In [ ]:
import itertools

issues = []

# Build ordered list of (year, month) tuples in the analysis window
def months_in_range(sy, sm, ey, em):
    result = []
    for y in range(sy, ey + 1):
        m_start = sm if y == sy else 1
        m_end   = em if y == ey else 12
        for m in range(m_start, m_end + 1):
            result.append((y, m))
    return result

if (START_YEAR, START_MONTH) > (END_YEAR, END_MONTH):
    issues.append(f"❌ Start ({START_YEAR}-{START_MONTH:02d}) is after end ({END_YEAR}-{END_MONTH:02d}).")

window_ym = months_in_range(START_YEAR, START_MONTH, END_YEAR, END_MONTH)

# Check data availability for each (year, month) in window
available_ym = set(zip(df['Year'], df['Month']))
missing_ym = [(y, m) for y, m in window_ym if (y, m) not in available_ym]
if missing_ym:
    issues.append(f"⚠️  No data for {len(missing_ym)} month(s) in window: {missing_ym[:5]}{'...' if len(missing_ym)>5 else ''}")

seasonal_lead_max = FORECAST_HORIZON_MONTHS * 480

if issues:
    for msg in issues: print(msg)

# Always print summary (warnings don't block execution)
print(f"✅ Inputs set!")
print(f"   Window   : {calendar.month_abbr[START_MONTH]} {START_YEAR} → {calendar.month_abbr[END_MONTH]} {END_YEAR}  ({len(window_ym)} months)")
print(f"   Forecast day     : {FORECAST_DAY}")
print(f"   Horizon (Plot 2) : {FORECAST_HORIZON_MONTHS} month(s) → seasonal lead max = {seasonal_lead_max}h")
print(f"   Plot 1 lead max  : {VALIDATION_LEAD_MAX}h (auto — unaffected by horizon)")


## 5. Volume Calculations

In [ ]:
def calc_volume(df, month, year, day, lead_max):
    mask = (
        (df['Month']     == month) &
        (df['Year']      == year)  &
        (df['Day']       == day)   &
        (df['lead_time'] <= lead_max)
    )
    subset = df[mask]
    if subset.empty:
        return None

    hf_cf  = subset['forecast'].sum()           * CFS_TO_CF_PER_STEP
    ref_cf = subset['reference_forecast'].sum() * CFS_TO_CF_PER_STEP
    obs_nan_count = subset['obs'].isna().sum()
    obs_cf = subset['obs'].fillna(0).sum()      * CFS_TO_CF_PER_STEP

    return dict(
        hf_cf=hf_cf,   ref_cf=ref_cf,   obs_cf=obs_cf,
        hf_af=hf_cf  / CF_TO_ACRE_FEET,
        ref_af=ref_cf / CF_TO_ACRE_FEET,
        obs_af=obs_cf / CF_TO_ACRE_FEET,
        obs_nan=obs_nan_count,
        n_steps=len(subset),
    )


# ── Validation volumes: 30-day, all months in window ─────────────────────────
val_volumes = {}   # keyed by (year, month)
for y, m in window_ym:
    r = calc_volume(df, m, y, FORECAST_DAY, VALIDATION_LEAD_MAX)
    if r:
        val_volumes[(y, m)] = r

# ── Seasonal volumes: FORECAST_HORIZON_MONTHS per issue month ────────────────
seasonal_volumes = {}
for y, m in window_ym:
    r = calc_volume(df, m, y, FORECAST_DAY, seasonal_lead_max)
    if r:
        seasonal_volumes[(y, m)] = r

window_with_data = [ym for ym in window_ym if ym in seasonal_volumes]
cumulative = {
    k: sum(seasonal_volumes[ym][k] for ym in window_with_data)
    for k in ('hf_af', 'ref_af', 'obs_af')
}

# ── Print check table ─────────────────────────────────────────────────────────
print(f"Step: {STEP_SIZE_HOURS}h | Plot 1 lead max: {VALIDATION_LEAD_MAX}h | Plot 2 lead max: {seasonal_lead_max}h")
print()
print(f"{'Month':<12} {'HF (cf)':>16} {'Ref (cf)':>16} {'Obs (cf)':>16}  {'Steps':>5}  {'NaNs':>5}")
print("-" * 74)
for ym in window_ym:
    if ym not in val_volumes: continue
    v = val_volumes[ym]
    y, m = ym
    nan_flag = " ⚠️" if v['obs_nan'] > 0 else ""
    print(f"{calendar.month_abbr[m]+' '+str(y):<12} {v['hf_cf']:>16,.0f} {v['ref_cf']:>16,.0f} "
          f"{v['obs_cf']:>16,.0f}  {v['n_steps']:>5}  {v['obs_nan']:>5}{nan_flag}")


## 6. Plot 1 — Validation: 30-Day Monthly Volumes (Cubic Feet)

Matches the Streamlit reference tool. Confirm bar heights align before using Plot 2 for analysis.
Months within the look-ahead window are highlighted in yellow.


In [ ]:
BAR_COLORS = {'HydroForecast': '#7BB8CC', 'Reference': '#C5A3D6', 'Observed': '#9E9E9E'}

def fmt_val(v):
    """Format a number to 4 sig figs with G/M/K suffix."""
    if v >= 1e9:  return f'{v/1e9:.4g}G'
    if v >= 1e6:  return f'{v/1e6:.4g}M'
    if v >= 1e3:  return f'{v/1e3:.4g}K'
    return f'{v:.4g}'

val_ym_sorted = sorted(val_volumes.keys())
xlabels = [calendar.month_abbr[m] + f' {y}' for y, m in val_ym_sorted]

series_cf = {
    'HydroForecast': [val_volumes[ym]['hf_cf']  for ym in val_ym_sorted],
    'Reference':     [val_volumes[ym]['ref_cf'] for ym in val_ym_sorted],
    'Observed':      [val_volumes[ym]['obs_cf'] for ym in val_ym_sorted],
}

fig1 = make_subplots(
    cols=2, column_widths=[0.78, 0.22],
    subplot_titles=(
        f'30-Day Forecasted Volumes — Day {FORECAST_DAY} of Each Month',
        'Sum of All Monthly Volumes',
    ),
)

for series, vals in series_cf.items():
    fmt = [fmt_val(v) for v in vals]
    fig1.add_trace(go.Bar(
        name=series, x=xlabels, y=vals,
        marker_color=BAR_COLORS[series], legendgroup=series,
        customdata=fmt,
        hovertemplate='%{x}<br>' + series + ': %{customdata}<extra></extra>',
    ), row=1, col=1)
    total_fmt = fmt_val(sum(vals))
    fig1.add_trace(go.Bar(
        name=series, x=['Sum of All Months'], y=[sum(vals)],
        marker_color=BAR_COLORS[series], legendgroup=series, showlegend=False,
        customdata=[total_fmt],
        hovertemplate='Sum<br>' + series + ': %{customdata}<extra></extra>',
    ), row=1, col=2)

# Highlight window months
for ym in window_with_data:
    y, m = ym
    lbl = calendar.month_abbr[m] + f' {y}'
    if lbl in xlabels:
        fig1.add_vrect(
            x0=xlabels.index(lbl) - 0.5, x1=xlabels.index(lbl) + 0.5,
            fillcolor='rgba(255,220,80,0.18)', line_width=0, row=1, col=1,
        )

window_label = f"{calendar.month_abbr[START_MONTH]} {START_YEAR} – {calendar.month_abbr[END_MONTH]} {END_YEAR}"
fig1.update_layout(
    title=dict(text=f'<b>Forecasted Volumes — {window_label}  [Validation]</b>', font_size=18),
    barmode='group', plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='v', x=1.02, y=1), height=520,
    yaxis=dict(title='Total Volume (Cubic Ft)', tickformat='.3s'),
    yaxis2=dict(tickformat='.3s'),
)
fig1.update_xaxes(showgrid=False)
fig1.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig1.show()
print(f"💡 Yellow = months included in Plot 2  |  lead_time ≤ {VALIDATION_LEAD_MAX}h (auto)  |  step = {STEP_SIZE_HOURS}h")


## 7. Plot 2 — Cumulative Volumes (Acre-Feet)

For each month in the window, this plot shows the total forecasted volume over the next
`FORECAST_HORIZON_MONTHS` months of lead time, expressed in acre-feet.
The right panel sums all months in the window into a single total.


In [ ]:
wm_labels = [calendar.month_abbr[m] + f' {y}' for y, m in window_with_data]
window_name = (f"{calendar.month_abbr[window_with_data[0][1]]} {window_with_data[0][0]}–"
               f"{calendar.month_abbr[window_with_data[-1][1]]} {window_with_data[-1][0]}"
               if len(window_with_data) > 1 else wm_labels[0])

series_af = {
    'HydroForecast': [seasonal_volumes[ym]['hf_af']  for ym in window_with_data],
    'Reference':     [seasonal_volumes[ym]['ref_af'] for ym in window_with_data],
    'Observed':      [seasonal_volumes[ym]['obs_af'] for ym in window_with_data],
}

fig2 = make_subplots(
    cols=2, column_widths=[0.78, 0.22],
    subplot_titles=(
        f'{FORECAST_HORIZON_MONTHS}-Month Cumulative Volumes — {window_name}',
        f'Total — {window_name}',
    ),
)

for series, vals in series_af.items():
    fmt = [fmt_val(v) for v in vals]
    fig2.add_trace(go.Bar(
        name=series, x=wm_labels, y=vals,
        marker_color=BAR_COLORS[series], legendgroup=series,
        customdata=fmt,
        hovertemplate='%{x}<br>' + series + ': %{customdata}<extra></extra>',
    ), row=1, col=1)
    total_fmt = fmt_val(sum(vals))
    fig2.add_trace(go.Bar(
        name=series, x=['Total'], y=[sum(vals)],
        marker_color=BAR_COLORS[series], legendgroup=series, showlegend=False,
        customdata=[total_fmt],
        hovertemplate='Total<br>' + series + ': %{customdata}<extra></extra>',
    ), row=1, col=2)

fig2.update_layout(
    title=dict(
        text=f'<b>Cumulative Volume — {FORECAST_HORIZON_MONTHS}-Month Horizon ({window_name})</b>',
        font_size=18,
    ),
    barmode='group', plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='v', x=1.02, y=1), height=520,
    yaxis=dict(title='Cumulative Volume (Acre-Feet)', tickformat='.3s'),
    yaxis2=dict(tickformat='.3s'),
)
fig2.update_xaxes(showgrid=False)
fig2.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.show()


## 8. HydroForecast % Improvement over Reference

Positive values indicate HydroForecast is **closer to observed** than the reference forecast.
Negative values indicate HydroForecast is **further from observed**.

`HF % Error = (HF − Obs) / Obs × 100`  
`Ref % Error = (Ref − Obs) / Obs × 100`  
`Improvement = |Ref % Error| − |HF % Error|`


In [ ]:
rows = []
for ym in window_with_data:
    v = seasonal_volumes[ym]
    y, m = ym
    hf, ref, obs = v['hf_af'], v['ref_af'], v['obs_af']
    if obs > 0:
        hf_err  = (hf  - obs) / obs * 100
        ref_err = (ref - obs) / obs * 100
        improvement = abs(ref_err) - abs(hf_err)
    else:
        hf_err = ref_err = improvement = float('nan')
    rows.append({
        'Month':              calendar.month_abbr[m] + f' {y}',
        'HydroForecast (af)': f'{hf:>12,.0f}',
        'Reference (af)':     f'{ref:>12,.0f}',
        'Observed (af)':      f'{obs:>12,.0f}',
        'HF % Error':         f'{hf_err:>+.1f}%'      if not pd.isna(hf_err)    else 'N/A',
        'Ref % Error':        f'{ref_err:>+.1f}%'     if not pd.isna(ref_err)   else 'N/A',
        'HF Improvement':     f'{improvement:>+.1f}%' if not pd.isna(improvement) else 'N/A',
    })

# Totals row
hf_t, ref_t, obs_t = cumulative['hf_af'], cumulative['ref_af'], cumulative['obs_af']
if obs_t > 0:
    hf_te  = (hf_t  - obs_t) / obs_t * 100
    ref_te = (ref_t - obs_t) / obs_t * 100
    imp_t  = abs(ref_te) - abs(hf_te)
else:
    hf_te = ref_te = imp_t = float('nan')

rows.append({
    'Month':              'TOTAL',
    'HydroForecast (af)': f'{hf_t:>12,.0f}',
    'Reference (af)':     f'{ref_t:>12,.0f}',
    'Observed (af)':      f'{obs_t:>12,.0f}',
    'HF % Error':         f'{hf_te:>+.1f}%'  if not pd.isna(hf_te)  else 'N/A',
    'Ref % Error':        f'{ref_te:>+.1f}%' if not pd.isna(ref_te) else 'N/A',
    'HF Improvement':     f'{imp_t:>+.1f}%'  if not pd.isna(imp_t)  else 'N/A',
})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


## 9. Download Plot 2

In [ ]:
plot2_filename = f'cumulative_volume_{window_name.replace(" ", "_").replace("–", "-")}.html'
fig2.write_html(plot2_filename)
files.download(plot2_filename)
print(f'✅ Downloaded {plot2_filename}')
